### 为什么需要事务？
事务把多个数据库操作包装成一个不可分割的业务单元：要么全部成功，要么全部失败。

全部成功 - commit
中途失败 - rollback

PostgreSQL 使用 BEGIN 和 COMMIT 包围一个事务；需要放弃时使用 ROLLBACK。

### commit是什么
表示：当前事务中的修改确认成功，可以正式提交。。

### rollback是什么
撤销当前事务中尚未提交的修改,数据库回到事务开始前的状态。

### ACID
是事务的四个核心性质，。
- Atomicity 原子性:要么全部成功，要么全部失败
- Consistency 一致性:事务执行前后，数据库状态保持一致
- Isolation 隔离性:并发事务之间相互隔离
- Durability 持久性:事务提交后，数据永久保存

### 事务不等于错误处理
开始事务 → 执行业务 → 成功：COMMIT → 异常：ROLLBACK → except 转换或继续抛出异常

> 数据库事务应尽可能短，只包住必须保持一致的数据库操作。

### 面试表达
> 事务将一组数据库操作作为一个原子业务单元执行，成功时统一提交，失败时统一回滚。例如转账中的扣款和加款必须位于同一事务中，否则服务在两步之间崩溃会产生部分成功。事务满足 ACID，但数据库事务只能直接管理数据库内的状态，无法自动撤销外部支付、邮件等副作用。

### 并发事务会出现什么问题？
#### 脏读
读到了别人还没提交的数据
#### 不可重复读
同一个事务中，重复读取同一行，却得到不同值。（值不同）
#### 幻读
同一个事务中，重复读取同一范围，却得到不同值。（集合不同）
#### 丢失更新
多个事务基于同一个旧值计算并写回，后写入者覆盖先写入者的结果。

#### 原子更新可以减少部分问题
直接让数据库执行相关计算操作，即原子更新。


### PostgreSQL 隔离级别
#### 什么事数据库快照？
当前事务允许看到的数据库版本。
#### Read Uncommitted
最低的隔离级别。可能允许脏读。但是在**postgresql中实际按照committed读取**。其他数据库不一定
#### Read Committed
PostgreSQL 默认隔离级别。每条 SQL 语句开始时，获取一份新的已提交数据快照。
- 不允许脏读
- 可能发生不可重复读
- 可能发生幻读
- 可能出现复杂并发异常
> 每条语句只读已提交数据，但下一条语句可能看到更新后的世界。

#### Repeatable Read
事务中的查询看到稳定的事务级快照。
```
事务A开始观察世界
→ 获得一张快照
→ 后续查询继续观察这张快照
```

在 PostgreSQL 中，Repeatable Read 比 SQL 标准要求更强：
- 不允许脏读
- 防止不可重复读
- PostgreSQL 中也防止幻读
- 仍可能发生序列化异常

#### Serializable
最严格的隔离级别,即使多个事务并发执行，最终效果也必须等价于它们按照某种顺序一个接一个执行。

Serializable 不一定把所有事务真的排队执行。PostgreSQL 会允许并发，但如果发现某组事务无法安全地解释成串行结果，就让其中某个事务失败。

Serializable 的常见代价是：
- 更多事务冲突
- 可能出现序列化失败
- 应用必须支持重试
- 长事务的冲突概率更高

### 三个级别的照片类比
#### Read committed
没执行一次select就重新拍一张照
#### Repeatable read
整个事务主要使用同一张照片
#### Serializable
不仅照片要合理，最终所有事务的共同结果还必须能解释称排队执行

### 隔离级别不是越高越无脑使用
普通 CRUD -> read committed
需要事务内稳定读取 -> repeatable read
复杂跨行规则 -> serializable

### 隔离级别简表

**Read Uncommitted**
- 快照理解：在 PostgreSQL 中按 Read Committed 处理
- 脏读：不会
- 不可重复读：可能
- 幻读：可能

**Read Committed**（默认）
- 快照理解：每条语句新快照
- 脏读：不会
- 不可重复读：可能
- 幻读：可能

**Repeatable Read**
- 快照理解：稳定事务快照
- 脏读：不会
- 不可重复读：不会
- 幻读：PostgreSQL 中不会

**Serializable**
- 快照理解：结果等价于串行执行
- 脏读：不会
- 不可重复读：不会
- 幻读：不会

记忆口令：
> PostgreSQL 从不脏读；默认每句换快照；RR 整事务同快照且防幻读；Serializable 要能解释成排队执行。

### FastAPI 中的事务、回滚与重试
1. Service 决定业务事务边界。
2. 失败后先回滚，再转换或抛出异常。
3. 并发重试要重跑整个业务事务，不是只重跑最后一条 SQL。

### 事务边界应该放在哪里？
```
Route
  ↓
Service：定义一次下单事务
  ├── Repository 查询库存
  ├── Repository 更新库存
  ├── Repository 保存订单
  └── Repository 保存明细
```

route负责：
- 接收请求
- 调用service
- 返回响应

Service负责：
- 业务规范
- 事务范围
- 多个repository的协调
- 冲突如何转换成业务错误

Repository
负责：
- 执行查询
- 插入、更新、删除
- 不决定整个下单业务何时提交

#### 数据库异常如何映射成 HTTP？
##### 唯一约束冲突
```
数据库 UNIQUE 冲突
→ 回滚
→ 转换成 EmailAlreadyExists
→ Exception Handler
→ HTTP 409 Conflict
```
##### 外键冲突
```
回滚
→ 转换成业务错误
→ 可能返回 409 或根据 API 语义返回 
```

##### 序列化失败或死锁

```
内部有限次数重试
→ 仍失败则返回服务冲突/暂时失败
```

### 面试
> 事务边界通常由 Service 根据业务用例定义，Repository 只负责具体数据访问。成功时提交，数据库异常时先回滚，再转换业务异常或继续抛出。对于序列化失败和死锁，应回滚并从头重试整个事务，因为之前读取的快照和业务判断可能已经失效。自动重试必须设置上限和退避，并保证外部副作用幂等。
